### Initialize instruments

In [ ]:
# import packages
from Equipments import BNC575, Weeder, SR400, MCBOX, Andor, LeCroyScope
from Equipments.MCBOX import generate_test_voltages
from DataProcessing.Plot import PlotSaver
from DataProcessing import Readout

import numpy as np
import pyvisa
import time
import matplotlib.pyplot as plt
# from scipy.signal import find_peaks

from matplotlib import rcParams
rcParams['font.family'] = 'Times New Roman'
rcParams['mathtext.fontset'] = 'stix'
rcParams['axes.unicode_minus'] = False
rcParams.update({
	'font.size': 14,
	'axes.titlesize': 14,
	'axes.labelsize': 14,
	'xtick.labelsize': 14,
	'ytick.labelsize': 14,
	'legend.fontsize': 14,
	'figure.titlesize': 14,
	'legend.frameon': False,
})

In [ ]:
# open list of available instruments
rm = pyvisa.ResourceManager()
print(rm.list_resources())

In [ ]:
# connect to BNC 575 pulse generator (for controlling experimental time sequence)
bnc575 = BNC575.BNC575("GPIB0::9::INSTR")

### Define timing sequence

In [ ]:
# use for testing
pulse_arrangement = [["B", 20e-3, 0, 2, 1],]
T = 40e-3
notes = {}

In [ ]:
notes["bnc575 T0"] = bnc575.clock_set(period = T, mode = "CONTINUOUS")
notes["bnc575 pulses"] = bnc575.pulse_sequence_setup(pulse_arrangement)

In [ ]:
bnc575.start_pulses()

### Collect oscilloscope traces

In [ ]:
# connect to LeCroy 9354AM oscilloscope (for displaying DEI pulse and Rydberg counts for both states)
lecroy = LeCroyScope.LeCroyScope('GPIB0::3::INSTR', timeout = 20000)
lecroy.setup_transfer()

In [ ]:
# collect trace for channel output (without offset)
channel = "A"
t_pulse, v_pulse, info = lecroy.read_waveform("C2")

plt.figure(1)
plt.plot(t_pulse, v_pulse)
plt.xlabel(info["horizontal_unit"] or "s")
plt.ylabel(info["vertical_unit"] or "V")
plt.title(f"Scope trace for channel {channel} (directly to scope)")
plt.grid(True)
plt.show()

In [ ]:
# collect trace for channel output (with offset)
channel = "A"
t_pulse, v_pulse, info = lecroy.read_waveform("C2")

plt.figure(1)
plt.plot(t_pulse, v_pulse)
plt.xlabel(info["horizontal_unit"] or "s")
plt.ylabel(info["vertical_unit"] or "V")
plt.title(f"Scope trace for channel {channel} (with offset)")
plt.grid(True)
plt.show()

### Shut down instrumentation

In [ ]:
# OPTIONAL: disables pulses from BNC box (can also be done by pressing RUN/STOP button or individual channel buttons on device)
bnc575.disable_all()

In [ ]:
# OPTIONAL: disarms the pulse generator (can also be done by pressing RUN/STOP button on device)
bnc575.disarm_all()

In [ ]:
# disconnect from pulse generator
bnc575.pyvisa.close()

In [ ]:
# run to disconnect from LeCroy (optional)
lecroy.close()